# Ekspor Gelombang-6 — TRAIN-ONLY utk 5 Kelas Lemah (perbesar lagi, fokus Tambang)

## Konteks
Gelombang-5 (`export_train_only_gel5.ipynb`) sudah menambah train-only utk 5 kelas lemah, tapi
hasil AKTUAL pasca-cut hanya **1.824 patch** (target ~5.952 patch agar pool minoritas gel.4+gel.5
mendekati 80:10:10) — rasio akhir baru **68,5% train** (target 80%, masih meleset >5pp).

**Keputusan (USER)**: tambah gelombang-6, **region lebih besar & lebih banyak**, fokus utama
Tambang (kelas yang paling sering gagal GATE di gelombang sebelumnya), pakai mega-tambang yang
sudah berproduksi sejak SEBELUM 2021 (supaya konsisten dgn vintage dataset Tang&Werner 2023 /
Maus 2022 yang dipakai `build_label()` utk kelas Tambang — lihat diagnosis di gelombang-5).
Target eksplisit: **minimal 4.000 patch train AKTUAL** dari gelombang-6 ini sendiri.

## Target volume (dihitung dari angka NYATA gel.4 + gel.5)
Pool minoritas (5 kelas lemah) gel.4+gel.5 AKTUAL saat ini: `train=4896 (3072+1824), val=1142, test=1114`
(val/test FIXED, tidak disentuh — sama seperti gelombang-5):

```
target_total = avg(val/0.10, test/0.10) = avg(11420, 11140) ~ 11280
target_train = target_total - val - test = 11280 - 1142 - 1114 = 9024
tambahan_train_dibutuhkan = 9024 - 4896 = 4128 patch BARU (train-only)
```

Floor eksplisit user: **>= 4.000 patch**. `TARGET_ADD_TRAIN_GEL6 = max(4128, 4000)` dipakai sbg
acuan GATE (bagian T2).

## Region gelombang-6 (22 region baru, 0 overlap dgn gelombang 1-5)
- **Tambang (9 region, PRIORITAS)**: mega-tambang berproduksi sejak sebelum 2021 di negara yang
  sudah terbukti reliable di dataset Tang&Werner/Maus (lihat gelombang-5) — Botswana (Jwaneng,
  berlian), Afrika Selatan (Witwatersrand, emas), Peru (Yanacocha, emas), Mongolia (Oyu Tolgoi,
  tembaga-emas), Zambia (Copperbelt), DR Congo (Kolwezi, kobalt-tembaga), Kanada (Sudbury,
  nikel), AS (Butte/Montana, tembaga — beroperasi sejak 1880-an), Australia (Olympic Dam,
  uranium-tembaga). Semua bbox diperbesar (~30-60 km sisi) dibanding gelombang-5 utk menutupi
  seluruh distrik tambang, bukan cuma satu pit.
- **Permukiman (4 region)**: kota menengah Indonesia yang belum pernah dipakai (Jambi, Ternate,
  Palangkaraya, Gorontalo), bbox diperbesar ke ~28-33 km supaya menangkap lebih banyak footprint
  urban.
- **Sawit, Lahan Terbuka, Pertanian Lain (3 region masing-masing)**: blok baru di Kalimantan
  Tengah, Jambi, Aceh, Sumsel, Jateng, Sulsel, Lampung — bbox diperbesar ke ~44-50 km (dari
  ~27-33 km di gelombang-5) utk volume lebih besar.

Estimasi total (offline, scale 100m): **~6.825 patch tile** (~6.000 AKTUAL pasca-cut dgn asumsi
konversi ~88% spt gelombang-5) — jauh di atas floor 4.000.

## Yang TIDAK disentuh
`Bahan_Training_Fix` (val/test Papua lama, FROZEN), `ForestWatch_Patches_TransferEval`
(val_new/test_new gelombang-4, FROZEN), `ForestWatch_Patches_TrainOnlyGel5` (train-only
gelombang-5, dipakai lagi sbg bagian gabungan), `forestwatch_papua_full_pipeline.ipynb`,
`train_model_1/2/3.ipynb`, `compare_and_select_best_model.ipynb`.

## Output akhir
Cell T8 menggabungkan SEMUA (Papua lama + gel.4 train/val/test + gel.5 train-only + gel.6
train-only), mencetak distribusi piksel final per split, cek rasio pool minoritas vs 80:10:10,
dan mem-paket ulang jadi `.tar` (format sama dgn `Bahan_Training_Fix`) ke
`Bahan_Training_Fix_Combined_v3/` — siap upload sbg Kaggle Dataset (superseded v2).


In [ ]:
# === COLAB SETUP (clone pertama kali / PULL sesi berikutnya) ===
%cd /content
!git -C fw_repo pull -q || git clone --depth 1 https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo
%cd fw_repo
!pip install -q -e ".[gee,gis,ml]"

import sys, importlib
if '/content/fw_repo/model/src' not in sys.path:
    sys.path.insert(0, '/content/fw_repo/model/src')
for mod in list(sys.modules.keys()):
    if mod == 'forestwatch' or mod.startswith('forestwatch.'):
        del sys.modules[mod]
importlib.invalidate_caches()

import forestwatch
print(f'forestwatch v{forestwatch.__version__}')


In [ ]:
# === MOUNT GOOGLE DRIVE ===
from google.colab import drive
drive.mount('/content/drive')
print('Drive ter-mount.')


In [ ]:
# === KONFIGURASI PATH (folder BARU, terpisah dari gelombang sebelumnya) ===
from pathlib import Path
from forestwatch.utils.io import save_json, load_json
from forestwatch.config import load_config

DRIVE_ROOT = Path('/content/drive/MyDrive/Satria Data 3.0')
TILES_TRAIN_GEL6  = DRIVE_ROOT / 'ForestWatch_Tiles_TrainOnlyGel6'
PATCHES_TRAIN_GEL6 = DRIVE_ROOT / 'ForestWatch_Patches_TrainOnlyGel6'
DIST_DIR = DRIVE_ROOT / 'Distribution_Reports'

for d in (TILES_TRAIN_GEL6, PATCHES_TRAIN_GEL6, DIST_DIR):
    d.mkdir(parents=True, exist_ok=True)

cfg = load_config()
print('Folder BARU (train-only gelombang-6):')
print(' -', TILES_TRAIN_GEL6)
print(' -', PATCHES_TRAIN_GEL6)


In [ ]:
# === AUTH GEE + Periode T2 ===
import ee
from forestwatch.gee.auth import init_ee

init_ee(project=cfg['project']['gee_project_id'])
T2 = cfg['periods']['t2']
print(f'GEE siap. Periode label T2={T2}.')


In [ ]:
# === Bagian T1 — TRAIN_REGIONS_GEL6 (region BARU, lebih besar, fokus Tambang) + cross-check GANDA ===
# Daftar eksklusi = SEMUA region gelombang 1-3 + gelombang-4 (EVAL_REGIONS) + gelombang-5
# (TRAIN_REGIONS_GEL5) -- gelombang-6 harus genuinely baru dari SEMUANYA.
ALREADY_USED_BBOXES = {
    # --- Tambang (gelombang 1-3) ---
    'kaltim_sangatta':        (117.31,   0.31, 117.79,   0.79),
    'kaltim_sangatta_gel3':   (117.40,   0.25, 117.90,   0.75),
    'kalsel_tanahbumbu':      (115.41,  -3.69, 115.89,  -3.21),
    'sulteng_morowali':       (121.946, -2.974, 122.234, -2.686),
    'sumbawa_batuhijau':      (116.729, -9.062, 117.001, -8.838),
    'chile_chuquicamata':     (-68.996, -22.416, -68.804, -22.224),
    'chile_escondida':        (-69.156, -24.369, -68.964, -24.161),
    'usa_bingham':            (-112.236, 40.440, -112.044, 40.600),
    'aus_kalgoorlie':         (121.414, -30.850, 121.606, -30.690),
    'aus_huntervalley':       (150.725, -32.686, 151.125, -32.334),
    'australia_huntervalley_gel3': (150.70, -32.75, 151.20, -32.25),
    'ger_hambach':            (6.412,   50.827,   6.668,  51.003),
    'halmahera_wedabay':      (127.62,  -0.75, 128.32,  -0.05),
    'aus_pilbara':            (119.35, -23.75, 120.15, -22.95),
    'usa_powderriver':        (-105.95,  43.85, -105.05,  44.75),
    'kalsel_adaro':           (115.10,  -2.50, 115.80,  -1.80),
    'brazil_carajas':         (-50.60,  -6.40, -49.90,  -5.70),
    'brazil_carajas_gel3':    (-50.40,  -6.20, -50.00,  -5.80),
    'papua_grasberg':         (137.00,  -4.30, 137.25,  -4.00),
    'peru_antamina':          (-77.15,  -9.62, -76.95,  -9.42),
    'safrica_witbank':        (29.10,  -26.00,  29.40,  -25.75),
    'sumsel_tanjungenim':     (103.65,  -3.90, 104.10,  -3.45),
    'safrica_sishen':         (22.70,  -28.10,  23.20,  -27.60),
    'canada_athabasca':       (-111.75,  56.95, -111.30,  57.40),
    'australia_bowenbasin':   (148.00, -22.00, 148.50, -21.50),
    # --- Permukiman (gelombang 1-3) ---
    'jabodetabek': (106.70, -6.35, 106.95, -6.10), 'bandung': (107.55, -6.97, 107.70, -6.85),
    'surabaya': (112.68, -7.32, 112.82, -7.20), 'medan': (98.62, 3.52, 98.74, 3.64),
    'makassar': (119.40, -5.18, 119.52, -5.08), 'palembang': (104.62, -3.08, 104.88, -2.85),
    'semarang': (110.32, -7.05, 110.52, -6.90), 'denpasar': (115.13, -8.75, 115.32, -8.55),
    'balikpapan': (116.78, -1.32, 116.98, -1.12), 'pekanbaru': (101.35, 0.42, 101.58, 0.62),
    'papua_jayapura': (140.66, -2.62, 140.78, -2.50), 'papua_merauke': (140.36, -8.52, 140.48, -8.40),
    'papua_timika': (136.84, -4.58, 136.96, -4.46), 'papua_sorong': (131.22, -0.92, 131.34, -0.80),
    'papua_biak': (136.04, -1.20, 136.16, -1.08), 'maluku_ambon': (128.14, -3.72, 128.26, -3.60),
    'ntt_kupang': (123.54, -10.22, 123.70, -10.08), 'bogor_depok': (106.65, -6.75, 107.05, -6.35),
    'padang': (100.20, -1.15, 100.60, -0.75), 'banjarmasin': (114.40, -3.50, 114.80, -3.10),
    'pontianak': (109.10, -0.20, 109.50, 0.20), 'manado': (124.65, 1.30, 125.05, 1.70),
    'yogyakarta': (110.25, -8.00, 110.60, -7.65),
    # --- Sawit (gelombang 1-3) ---
    'riau_pelalawan': (101.40, 0.20, 101.70, 0.50), 'sumut': (99.50, 2.00, 99.80, 2.30),
    'kalbar': (109.50, 0.00, 109.80, 0.30), 'kalteng': (112.50, -2.20, 112.80, -1.90),
    'sumsel_musibanyuasin': (103.80, -2.85, 104.20, -2.45),
    'riau_rokanhilir': (100.90, 1.50, 101.30, 1.90), 'riau_kampar': (101.00, 0.00, 101.40, 0.40),
    'riau_indragirihilir': (102.80, -0.70, 103.20, -0.30),
    'sumut_labuhanbatu': (99.90, 1.85, 100.30, 2.25), 'kalbar_ketapang': (110.20, -1.80, 110.60, -1.40),
    # --- Lahan Terbuka (gelombang 1-3) ---
    'kaltim_tepitambang': (117.30, 0.30, 117.55, 0.55), 'kalteng_pascabakar': (113.50, -2.50, 113.80, -2.20),
    # --- Pertanian Lain (gelombang 1-3) ---
    'pantura_indramayu': (108.20, -6.45, 108.45, -6.25), 'lampung_ladang': (105.20, -5.10, 105.45, -4.90),
    # --- Gelombang-4 (export_eval_only_transfer.ipynb) -- SEMUA kelas ---
    'chile_lospelambres': (-70.65, -31.85, -70.35, -31.60), 'peru_cerroverde': (-71.65, -16.62, -71.42, -16.42),
    'usa_morenci': (-109.48, 32.98, -109.25, 33.18), 'australia_mountisa': (139.40, -20.80, 139.62, -20.62),
    'chile_losbronces': (-70.40, -33.25, -70.20, -33.05), 'usa_climax': (-106.30, 39.28, -106.05, 39.48),
    'peru_toquepala_cuajone': (-70.85, -17.30, -70.50, -16.95), 'indonesia_sorowako': (121.20, -2.65, 121.55, -2.35),
    'cirebon': (108.40, -6.90, 108.75, -6.60), 'malang': (112.50, -8.05, 112.75, -7.85),
    'tasikmalaya': (108.10, -7.45, 108.35, -7.25), 'manokwari': (133.95, -0.95, 134.20, -0.75),
    'aceh_acehtimur': (97.50, 4.20, 98.10, 4.80), 'sulbar_pasangkayu': (119.20, -1.40, 119.80, -0.80),
    'riau_tepigambut': (101.90, -0.20, 102.75, 0.70), 'ntb_sumbawa_pascabakar': (117.20, -8.85, 117.90, -8.15),
    'jatim_ladang_v2': (111.60, -7.95, 112.30, -7.55), 'bali_tabanan': (114.95, -8.55, 115.35, -8.15),
    # --- Gelombang-5 (export_train_only_gel5.ipynb) -- SEMUA kelas ---
    'usa_raymine':       (-111.05, 32.95, -110.80, 33.20), 'mexico_cananea':    (-110.40, 30.85, -110.15, 31.10),
    'chile_collahuasi':  (-68.80, -21.10, -68.50, -20.80), 'peru_lasbambas':    (-72.58, -14.10, -72.28, -13.85),
    'australia_cadia':   (148.90, -33.60, 149.20, -33.30),
    'palu':    (119.82, -1.05, 120.02, -0.78), 'kendari': (122.45, -4.05, 122.65, -3.85),
    'wamena':  (138.85, -4.20, 139.05, -4.00),
    'sumut_asahan':   (99.70,  2.80, 100.00, 3.10), 'kalbar_sanggau': (110.50, 0.00, 110.80, 0.30),
    'riau_siak':      (101.80, 0.75, 102.10, 1.05),
    'riau_bengkalis': (102.00,  1.20, 102.30,  1.50), 'sumsel_ogan':    (104.00, -3.40, 104.30, -3.10),
    'jabar_subang':       (107.75, -6.70, 108.05, -6.40), 'lampung_pesawaran':  (105.05, -5.45, 105.35, -5.15),
}

ALREADY_USED = {
    'tambang': {
        'kaltim_sangatta', 'kalsel_tanahbumbu', 'sulteng_morowali', 'sumbawa_batuhijau',
        'chile_chuquicamata', 'chile_escondida', 'usa_bingham', 'aus_kalgoorlie',
        'aus_huntervalley', 'ger_hambach', 'halmahera_wedabay', 'aus_pilbara',
        'usa_powderriver', 'kalsel_adaro', 'brazil_carajas', 'papua_grasberg',
        'peru_antamina', 'safrica_witbank', 'sumsel_tanjungenim', 'safrica_sishen',
        'canada_athabasca', 'australia_bowenbasin', 'australia_huntervalley', 'kaltim_paser',
        'chile_lospelambres', 'peru_cerroverde', 'usa_morenci', 'australia_mountisa',
        'chile_losbronces', 'usa_climax', 'peru_toquepala_cuajone', 'indonesia_sorowako',
        'usa_raymine', 'mexico_cananea', 'chile_collahuasi', 'peru_lasbambas', 'australia_cadia',
    },
    'permukiman': {
        'jabodetabek', 'bandung', 'surabaya', 'medan', 'makassar', 'palembang', 'semarang',
        'denpasar', 'balikpapan', 'pekanbaru', 'papua_jayapura', 'papua_merauke',
        'papua_timika', 'papua_sorong', 'papua_biak', 'maluku_ambon', 'ntt_kupang',
        'bogor_depok', 'padang', 'banjarmasin', 'pontianak', 'manado', 'yogyakarta',
        'cirebon', 'malang', 'tasikmalaya', 'manokwari', 'palu', 'kendari', 'wamena',
    },
    'sawit': {
        'riau_pelalawan', 'sumut', 'kalbar', 'kalteng', 'sumsel_musibanyuasin',
        'riau_rokanhilir', 'riau_kampar', 'riau_indragirihilir', 'sumut_labuhanbatu',
        'kalbar_ketapang', 'jambi_tebo', 'kalsel_kotabaru', 'aceh_acehtimur', 'sulbar_pasangkayu',
        'sumut_asahan', 'kalbar_sanggau', 'riau_siak',
    },
    'lahan_terbuka': {
        'kaltim_tepitambang', 'kalteng_pascabakar', 'riau_tepigambut', 'ntb_sumbawa_pascabakar',
        'riau_bengkalis', 'sumsel_ogan',
    },
    'pertanian_lain': {
        'pantura_indramayu', 'lampung_ladang', 'jatim_ladang_v2', 'bali_tabanan',
        'jabar_subang', 'lampung_pesawaran',
    },
}

# Region BARU -- bbox diperbesar dibanding gelombang-5 (lihat markdown atas) supaya volume
# AKTUAL pasca-cut tembus floor 4.000 patch. Tambang diprioritaskan: 9 mega-tambang yang sudah
# berproduksi SEBELUM 2021 (vintage Tang&Werner 2023 / Maus 2022) di negara yg sudah reliable.
TRAIN_REGIONS_GEL6 = {
    'tambang': [
        ('botswana_jwaneng',       (24.50, -24.75, 24.90, -24.35)),   # berlian, sejak 1982
        ('safrica_witwatersrand',  (27.65, -26.45, 28.25, -25.95)),   # emas, sejak 1880-an
        ('peru_yanacocha',         (-78.70, -7.05, -78.30, -6.65)),   # emas, sejak 1993
        ('mongolia_oyutolgoi',     (106.75, 42.95, 107.15, 43.25)),   # tembaga-emas, sejak 2013 (eksplorasi 2001)
        ('zambia_copperbelt',      (28.40, -13.00, 28.90, -12.50)),   # tembaga, sejak awal abad-20
        ('drc_kolwezi',            (25.30, -10.90, 25.80, -10.40)),   # kobalt-tembaga, sejak 1930-an
        ('canada_sudbury',         (-81.20, 46.30, -80.70, 46.70)),   # nikel, sejak 1880-an
        ('usa_butte',              (-112.70, 45.90, -112.30, 46.30)),  # tembaga, sejak 1880-an
        ('australia_olympicdam',   (136.70, -30.55, 137.10, -30.15)),  # uranium-tembaga, sejak 1988
    ],
    'permukiman': [
        ('jambikota',      (103.45, -1.75, 103.75, -1.45)),
        ('ternate',        (127.25, 0.65, 127.50, 0.95)),
        ('palangkaraya',   (113.80, -2.35, 114.10, -2.05)),
        ('gorontalokota',  (122.95, 0.45, 123.20, 0.70)),
    ],
    'sawit': [
        ('kalteng_kotawaringin', (111.35, -3.00, 111.80, -2.55)),
        ('jambi_merangin',       (102.15, -2.35, 102.60, -1.90)),
        ('aceh_nagan',           (96.25, 3.95, 96.70, 4.40)),
    ],
    'lahan_terbuka': [
        ('kalteng_pulangpisau', (113.80, -3.10, 114.20, -2.70)),
        ('sumsel_oki',          (104.95, -3.45, 105.40, -3.00)),
        ('jambi_tanjabar',      (103.30, -1.25, 103.75, -0.80)),
    ],
    'pertanian_lain': [
        ('jateng_cilacap',          (108.85, -7.90, 109.30, -7.45)),
        ('sulsel_bone',             (120.40, -4.70, 120.85, -4.25)),
        ('lampung_tulangbawang',    (105.50, -4.55, 105.95, -4.10)),
    ],
}


def _bbox_overlap(a, b):
    """True bila 2 bbox (lon0,lat0,lon1,lat1) berpotongan (area > 0)."""
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return ax0 < bx1 and bx0 < ax1 and ay0 < by1 and by0 < ay1


print('=== Cross-check 1: nama region vs ALREADY_USED (gelombang 1-5) ===')
collisions = []
for slug, regions in TRAIN_REGIONS_GEL6.items():
    for name, bbox in regions:
        if name in ALREADY_USED.get(slug, set()):
            collisions.append((slug, name))
assert not collisions, f'Nama region COLLIDE dengan yang sudah dipakai: {collisions}'
print('OK -- tidak ada nama region yang sama dengan gelombang 1-5.')

print('\n=== Cross-check 2: OVERLAP KOORDINAT bbox vs SEMUA bbox terpakai (lintas kelas) ===')
geo_overlaps = []
for slug, regions in TRAIN_REGIONS_GEL6.items():
    for name, bbox in regions:
        for used_name, used_bbox in ALREADY_USED_BBOXES.items():
            if _bbox_overlap(bbox, used_bbox):
                geo_overlaps.append((slug, name, used_name))
assert not geo_overlaps, (
    f'Region BARU overlap geografis dgn region yg sudah dipakai (leakage risk): {geo_overlaps}')
print(f'OK -- {sum(len(r) for r in TRAIN_REGIONS_GEL6.values())} region baru TIDAK overlap koordinat '
      f'dgn {len(ALREADY_USED_BBOXES)} bbox yang sudah pernah diekspor (gelombang 1-5).')

print('\n=== Cross-check 3: OVERLAP internal antar region gelombang-6 sendiri ===')
all_new = [(s, n, b) for s, rs in TRAIN_REGIONS_GEL6.items() for n, b in rs]
internal_overlaps = []
for i in range(len(all_new)):
    for j in range(i + 1, len(all_new)):
        s1, n1, b1 = all_new[i]
        s2, n2, b2 = all_new[j]
        if _bbox_overlap(b1, b2):
            internal_overlaps.append((s1, n1, s2, n2))
assert not internal_overlaps, f'Region BARU overlap dgn region BARU lain: {internal_overlaps}'
print(f'OK -- {len(all_new)} region baru tidak overlap satu sama lain.')

import math
print()
for slug, regions in TRAIN_REGIONS_GEL6.items():
    print(f'{slug}:')
    for name, bbox in regions:
        x0, y0, x1, y1 = bbox
        w_km = (x1 - x0) * 111 * math.cos(math.radians((y0 + y1) / 2))
        h_km = (y1 - y0) * 111
        print(f'  {name:<24}: {bbox}  (~{w_km:.0f} x {h_km:.0f} km)')


In [ ]:
# === Bagian T2 — Preview + GATE: pastikan region BARU mengandung kelas targetnya + cek target volume ===
import numpy as np
from forestwatch.gee.composite import s2_composite
from forestwatch.gee.label_fusion import build_label
from forestwatch.constants import CLASS_NAMES, CLASS_SLUGS, N_CLASSES, DEG_TO_M

EST_SCALE = 100
MIN_OWN_FRAC = 0.01
SLUG_TO_CLASS = {v: k for k, v in enumerate(CLASS_SLUGS)}
PATCH_PX = cfg['patches']['size'] ** 2

# Target dihitung dari angka NYATA gel.4+gel.5 (lihat markdown atas) -- val/test FIXED, floor
# eksplisit 4.000 patch (instruksi USER) dipakai sbg lantai minimum.
GEL45_TRAIN, GEL45_VAL, GEL45_TEST = 4896, 1142, 1114
TARGET_TOTAL_POOL = round((GEL45_VAL / 0.10 + GEL45_TEST / 0.10) / 2)
TARGET_TRAIN_POOL = TARGET_TOTAL_POOL - GEL45_VAL - GEL45_TEST
TARGET_ADD_TRAIN_GEL6 = max(4000, TARGET_TRAIN_POOL - GEL45_TRAIN)
print(f'Target tambahan train-only gelombang-6: ~{TARGET_ADD_TRAIN_GEL6:,} patch '
      f'(supaya pool gel.4+gel.5+gel.6 mendekati 80:10:10, floor eksplisit 4.000 dari USER).')
TARGET_ADD_PATCH_PER_CLASS = TARGET_ADD_TRAIN_GEL6 / 5   # 5 kelas lemah, target rata kasar

print(f'GATE pre-export gelombang-6 (scale {EST_SCALE} m)...\n')
region_reports = []
gate = {}
agg_est_px = {c: 0.0 for c in range(N_CLASSES)}
agg_est_patch_per_class = {c: 0.0 for c in range(N_CLASSES)}
for slug, regions in TRAIN_REGIONS_GEL6.items():
    own_cls = SLUG_TO_CLASS[slug]
    for name, bbox in regions:
        x0, y0, x1, y1 = bbox
        w_m = (x1 - x0) * DEG_TO_M * np.cos(np.radians((y0 + y1) / 2))
        h_m = (y1 - y0) * DEG_TO_M
        area_px = (w_m / 10) * (h_m / 10)
        box = ee.Geometry.Rectangle(list(bbox))
        img = s2_composite(T2, box)
        label = build_label(box, T2, composite=img)
        fh = label.reduceRegion(ee.Reducer.frequencyHistogram(), box, EST_SCALE,
                                 maxPixels=int(1e13)).getInfo().get('label', {})
        cnt = {c: float(fh.get(str(c), fh.get(f'{c}.0', 0)) or 0) for c in range(N_CLASSES)}
        tot = sum(cnt.values()) or 1.0
        own_frac = cnt[own_cls] / tot
        region_reports.append({
            'slug': slug, 'name': name, 'area_px_native': area_px,
            'own_class': CLASS_NAMES[own_cls], 'own_class_frac': own_frac,
            'own_class_px_est': area_px * own_frac,
        })
        key = f'{slug}/{name}: >={MIN_OWN_FRAC * 100:.0f}% piksel {CLASS_NAMES[own_cls]}'
        gate[key] = own_frac >= MIN_OWN_FRAC
        print(f'  {slug:<16}/{name:<24}: {CLASS_NAMES[own_cls]:<14} frac={own_frac * 100:5.1f}% '
              f'(~{area_px * own_frac / 1e6:5.2f} jt px, ~{int(area_px / PATCH_PX):,} patch tile)')
        for c in range(N_CLASSES):
            agg_est_px[c] += area_px * (cnt[c] / tot)
        agg_est_patch_per_class[own_cls] += area_px / PATCH_PX

print()
print('=== GATE T2: kelayakan region ===')
for k, v in gate.items():
    print(f"  [{'OK ' if v else 'X  '}] {k}")

print('\n=== Estimasi volume TRAIN-ONLY vs target (per kelas, ~{:.0f} patch/kelas) ==='.format(
    TARGET_ADD_PATCH_PER_CLASS))
total_est = 0.0
for c in (2, 3, 4, 5, 6):
    est = agg_est_patch_per_class[c]
    total_est += est
    pct = 100 * est / TARGET_ADD_PATCH_PER_CLASS if TARGET_ADD_PATCH_PER_CLASS else 0
    print(f'  {CLASS_NAMES[c]:<16}: ~{est:>8,.0f} patch tile (estimasi)  ({pct:5.1f}% target)')
print(f'\nTotal estimasi patch tile (semua kelas): ~{total_est:,.0f} '
      f'(floor user 4.000 -> {"AMAN, di atas floor" if total_est >= 4000 else "MASIH KURANG, tambah region"})')
print('\nCatatan: ini estimasi LUAS TILE (bukan jumlah patch valid pasca-cut -- ada patch yg')
print('dibuang krn NaN/laut/awan). Angka AKTUAL baru diketahui di T7. Kalau jauh di bawah target,')
print('tambah region lagi di T1 (pola aditif yg sama) sebelum lanjut export.')

save_json({'scale_m': EST_SCALE, 'min_own_frac': MIN_OWN_FRAC, 'regions': region_reports,
           'gate': gate, 'passed': all(gate.values()),
           'target_add_train_gel6': TARGET_ADD_TRAIN_GEL6,
           'agg_estimate_px': {CLASS_NAMES[c]: agg_est_px[c] for c in range(N_CLASSES)}},
          DIST_DIR / 'pixel_estimate_gel6_train.json')

for c in (2, 3, 4, 5, 6):
    assert agg_est_px[c] > 0, f'{CLASS_NAMES[c]} estimasi 0 px -- region gagal, revisi T1.'
n_weak = sum(1 for v in gate.values() if not v)
if n_weak:
    print(f'\n*** PERINGATAN: {n_weak} region <1% kelas targetnya -- review/ganti bbox di T1. ***')
else:
    print('\nGATE LULUS -- semua region gelombang-6 relevan dgn kelas targetnya. Lanjut ke T3.')


In [ ]:
# === Bagian T3 — Export GEE (additif, folder FLAT per-slug train6_<slug>) ===
import math

assert all(_v for _v in __import__('json').load(open(DIST_DIR / 'pixel_estimate_gel6_train.json'))['gate'].values()), (
    'GATE T2 belum lulus semua -- revisi region di T1 dulu.')
from forestwatch.gee.tiles import make_tiles
from forestwatch.gee.export import export_tiles_grid

TARGET_KM_PER_TILE = 25

train6_tasks = []
for slug, regions in TRAIN_REGIONS_GEL6.items():
    for name, bbox in regions:
        x0, y0, x1, y1 = bbox
        w_km = (x1 - x0) * 111 * math.cos(math.radians((y0 + y1) / 2))
        h_km = (y1 - y0) * 111
        nx = max(2, round(w_km / TARGET_KM_PER_TILE))
        ny = max(2, round(h_km / TARGET_KM_PER_TILE))

        box = ee.Geometry.Rectangle(list(bbox))
        img = s2_composite(T2, box)
        label = build_label(box, T2, composite=img)
        stack = img.addBands(label.toFloat())
        tiles = make_tiles(box, nx=nx, ny=ny)
        tasks = export_tiles_grid(
            stack, tiles,
            name_prefix=f'train6_{slug}_{name}_tile',
            folder=f'train6_{slug}',
            scale=cfg['sentinel2']['scale'],
            max_pixels=int(cfg['export']['max_pixels']),
        )
        train6_tasks.extend(tasks)
        print(f'  {slug:<16}/{name:<24}: {w_km:.0f}x{h_km:.0f} km -> grid {nx}x{ny} = {nx*ny} tile')

print(f'\n{len(train6_tasks)} task ekspor gelombang-6 (train-only) disubmit -> folder Drive train6_<slug>/.')
print('Pantau: https://code.earthengine.google.com/tasks')
print('Tunggu SEMUA task COMPLETED sebelum lanjut ke T4 (pindah tile nyasar + cut patches).')


In [ ]:
# === Bagian T4 — Pindahkan tile yang nyasar, FLAT per-slug ===
import shutil
for slug in TRAIN_REGIONS_GEL6:
    src = DRIVE_ROOT / 'Augmented_Patches' / f'train6_{slug}'
    dst = TILES_TRAIN_GEL6 / slug
    dst.mkdir(parents=True, exist_ok=True)
    if src.exists():
        moved = 0
        for f in src.glob('*.tif'):
            shutil.move(str(f), str(dst / f.name))
            moved += 1
        if moved:
            print(f'{slug}: {moved} file dipindah -> {dst}')
print('Selesai cek tile nyasar.')


In [ ]:
# === Bagian T5 — cut_patches_resilient (disalin verbatim dari gelombang-5) ===
import numpy as np
import rasterio
import shutil
import time
from rasterio.windows import Window
from pathlib import Path
from tqdm.auto import tqdm


def _remount_drive():
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(3)
    drive.mount('/content/drive', force_remount=True)
    time.sleep(2)
    print("  -> Drive di-remount.")


def _read_done_count(done_marker):
    try:
        if done_marker.exists():
            return int(done_marker.read_text().strip())
    except (OSError, ValueError):
        return None
    return None


def cut_patches_resilient(tile_dir, patch_base_dir, *,
                          patch_size=256, stride=256, max_nan_ratio=0.3,
                          n_channels_image=6, local_tmp='/content/_tmp_patches'):
    tile_files = sorted(Path(tile_dir).glob('*.tif'))
    if not tile_files:
        raise FileNotFoundError(f"Tidak ada .tif di {tile_dir}")

    patch_base = Path(patch_base_dir)
    local_root = Path(local_tmp)
    n_tiles = len(tile_files)
    total = 0

    for ti, tif in enumerate(tile_files):
        drive_out = patch_base / f'tile_{ti:03d}'
        done_marker = drive_out / '_DONE'
        header = f"Tile {ti+1}/{n_tiles}  (tile_{ti:03d})"

        try:
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0
        except OSError:
            _remount_drive()
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0

        if done_n is not None and actual_n == done_n:
            total += actual_n
            tag = "laut/kosong" if done_n == 0 else f"{actual_n} patch"
            print(f"{header}: SKIP - komplit ({tag})")
            continue

        try:
            if drive_out.exists():
                shutil.rmtree(drive_out, ignore_errors=True)
        except OSError:
            _remount_drive()
            shutil.rmtree(drive_out, ignore_errors=True)

        ltile = local_root / f'tile_{ti:03d}'
        if ltile.exists():
            shutil.rmtree(ltile)
        ltile.mkdir(parents=True, exist_ok=True)

        idx = 0
        with rasterio.open(tif) as src:
            W, H, nb = src.width, src.height, src.count
            row_list = list(range(0, H - patch_size + 1, stride))
            col_list = list(range(0, W - patch_size + 1, stride))
            pbar = tqdm(total=len(row_list) * len(col_list), desc=header, unit="win", leave=True)
            for r in row_list:
                for c in col_list:
                    arr = src.read(window=Window(c, r, patch_size, patch_size))
                    pbar.update(1)
                    if arr.shape != (nb, patch_size, patch_size):
                        continue
                    img = arr[:n_channels_image].astype('float32')
                    if np.isnan(img).mean() > max_nan_ratio:
                        continue
                    img = np.nan_to_num(img)
                    kw = dict(img=img, tile=tif.name, row=r, col=c)
                    if nb > n_channels_image:
                        kw['lab'] = np.nan_to_num(arr[n_channels_image], nan=0.0).astype('uint8')
                    np.savez_compressed(ltile / f'p{idx:05d}.npz', **kw)
                    idx += 1
                    pbar.set_postfix(patch=idx)
            pbar.close()

        local_files = sorted(ltile.glob('p*.npz'))
        assert len(local_files) == idx, "jumlah file lokal tidak konsisten"

        synced = False
        for attempt in range(1, 8):
            try:
                drive_out.mkdir(parents=True, exist_ok=True)
                for f in local_files:
                    dst = drive_out / f.name
                    if (not dst.exists()) or (dst.stat().st_size != f.stat().st_size):
                        shutil.copy2(f, dst)
                drive_n = len(list(drive_out.glob('p*.npz')))
                if drive_n == idx:
                    done_marker.write_text(str(idx))
                    synced = True
                    break
                print(f"  verifikasi belum cocok: Drive={drive_n} vs lokal={idx} (attempt {attempt})")
            except OSError as e:
                print(f"  sync gagal (attempt {attempt}): {e}")
                _remount_drive()
            time.sleep(2)
        shutil.rmtree(ltile, ignore_errors=True)

        if not synced:
            print(f"{header}: GAGAL verifikasi sync - STOP. Hapus folder tile ini di Drive lalu jalankan ulang.")
            return total

        total += idx
        tag = "LAUT/kosong (0 patch)" if idx == 0 else f"{idx} patch"
        print(f"{header}: SELESAI - {tag} -> Drive/tile_{ti:03d}/\n")

    print(f"=== SEMUA TILE SELESAI. Total {total} patch ===")
    return total


print("cut_patches_resilient siap.")


In [ ]:
# === Bagian T6 — Jalankan cut patches FLAT per-slug (FASE 2, jalankan SETELAH semua task T3 COMPLETED) ===
from forestwatch.data.patches import list_patches

manifest_gel6 = {'source': 'train_only_gel6', 'classes': {}}
for slug in TRAIN_REGIONS_GEL6:
    tile_dir  = TILES_TRAIN_GEL6 / slug
    patch_dir = PATCHES_TRAIN_GEL6 / slug
    n_tif = len(sorted(tile_dir.glob('*.tif'))) if tile_dir.exists() else 0
    if n_tif == 0:
        print(f'  [skip] {slug}: belum ada .tif di {tile_dir} (task T3 selesai?).')
        continue
    cut_patches_resilient(tile_dir, patch_dir,
                          patch_size=cfg['patches']['size'], stride=cfg['patches']['size'])
    n_patch = len(list_patches(patch_dir))
    manifest_gel6['classes'][slug] = {'n_tif': int(n_tif), 'n_patch': int(n_patch)}
    print(f'  [ok] {slug}: {n_tif} tif -> {n_patch} patch')

save_json(manifest_gel6, DIST_DIR / 'train_only_gel6_export_manifest.json')
print('FASE 2 (cut) selesai. Manifest -> Distribution_Reports/train_only_gel6_export_manifest.json')


In [ ]:
# === Bagian T7 — Verifikasi AKTUAL pasca-cut: distribusi piksel + cek vs target volume ===
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from forestwatch.constants import CLASS_NAMES, N_CLASSES

train_new_gel6 = list_patches(PATCHES_TRAIN_GEL6)
print(f'Total patch gelombang-6 train-only (semua kelas gabung): {len(train_new_gel6)}')
print(f'Target tambahan (dari T2, floor 4.000): ~{TARGET_ADD_TRAIN_GEL6:,} patch')
pct_of_target = 100 * len(train_new_gel6) / TARGET_ADD_TRAIN_GEL6 if TARGET_ADD_TRAIN_GEL6 else 0
print(f'Capaian: {pct_of_target:.1f}% dari target.')
print(f'Floor eksplisit USER (4.000 patch): {"TERPENUHI" if len(train_new_gel6) >= 4000 else "BELUM TERPENUHI -- tambah region lagi"}.')


def _read_lab(f):
    return np.load(f)['lab']


dist_gel6 = {c: 0 for c in range(N_CLASSES)}
with ThreadPoolExecutor(max_workers=32) as exe:
    for lab in tqdm(exe.map(_read_lab, train_new_gel6), total=len(train_new_gel6),
                    desc='Distribusi gel.6 train-only'):
        u, cnt = np.unique(lab, return_counts=True)
        for cls, n in zip(u.tolist(), cnt.tolist()):
            if 0 <= int(cls) < N_CLASSES:
                dist_gel6[int(cls)] += int(n)

print('\nDistribusi piksel TRAIN-ONLY GELOMBANG-6 (gabungan, AKTUAL pasca-cut):')
for c in range(N_CLASSES):
    print(f'  {CLASS_NAMES[c]:<16}: {dist_gel6[c]:>12,} px')

save_json({'counts': {str(c): dist_gel6[c] for c in range(N_CLASSES)},
           'n_patch': len(train_new_gel6), 'target_add_train_gel6': TARGET_ADD_TRAIN_GEL6},
          DIST_DIR / 'distribution_gel6_train.json')
print('\nVerifikasi kelima kelas (2-6) HARUS > 0 sebelum lanjut ke T8:')
for c in (2, 3, 4, 5, 6):
    assert dist_gel6[c] > 0, (
        f'{CLASS_NAMES[c]} = 0 piksel di gelombang-6 -- region utk kelas ini gagal, revisi T1/T2.')
print('OK -- kelima kelas (2-6) terbukti punya piksel NYATA di gelombang-6 train-only.')


In [ ]:
# === Bagian T8 — Gabung FINAL: Papua lama + gel.4 (train/val/test) + gel.5 + gel.6 (train-only) ===
# Distribusi FIX totalnya + cek rasio pool minoritas (5 kelas lemah, gel.4+gel.5+gel.6) vs
# 80:10:10 + paket ulang .tar siap Kaggle (v3 -- mengganti Bahan_Training_Fix_Combined v2).
from forestwatch.data.dataset import extract_dataset_archives, create_dataset_archives

# --- Muat gelombang-4 (train_new/val_new/test_new) dari manifest -- TIDAK diubah ---
_m4 = load_json(DIST_DIR / 'split_manifest_gel4_eval.json')
_base4 = Path(_m4['base_dir'])
gel4_train = [_base4 / r for r in _m4['train_new']]
gel4_val   = [_base4 / r for r in _m4['val_new']]
gel4_test  = [_base4 / r for r in _m4['test_new']]

# --- Muat gelombang-5 (train-only) -- TIDAK diubah ---
train_new_gel5 = list_patches(PATCHES_TRAIN_GEL5) if 'PATCHES_TRAIN_GEL5' in dir() else list_patches(
    DRIVE_ROOT / 'ForestWatch_Patches_TrainOnlyGel5')

# --- Cek rasio pool MINORITAS SAJA (gel.4 + gel.5 + gel.6 train-only) vs 80:10:10 ---
pool_train = len(gel4_train) + len(train_new_gel5) + len(train_new_gel6)
pool_val, pool_test = len(gel4_val), len(gel4_test)
pool_tot = pool_train + pool_val + pool_test
print('=== Rasio pool MINORITAS (gelombang-4 + gelombang-5 + gelombang-6, 5 kelas lemah) ===')
print(f'  train: {pool_train:>7,} ({100 * pool_train / pool_tot:5.1f}%)  -- target 80%')
print(f'  val  : {pool_val:>7,} ({100 * pool_val / pool_tot:5.1f}%)  -- target 10%')
print(f'  test : {pool_test:>7,} ({100 * pool_test / pool_tot:5.1f}%)  -- target 10%')
_train_pct = 100 * pool_train / pool_tot
if not (75 <= _train_pct <= 85):
    print(f'*** PERINGATAN: rasio train pool minoritas {_train_pct:.1f}% -- masih meleset >5pp dari '
          f'80%. Tambah region lagi (gelombang-7) kalau perlu lebih presisi. ***')
else:
    print('OK -- rasio pool minoritas dalam toleransi +-5pp dari 80:10:10.')

# --- Muat Papua lama (FROZEN, val/test TIDAK disentuh) ---
BAHAN_DIR = DRIVE_ROOT / 'Bahan_Training_Fix'
LOCAL_OLD = Path('/content/dataset_local_old')
old_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_OLD, splits=('train', 'val', 'test'), max_workers=8)
old_train = list_patches(old_dirs['train'])
old_val   = list_patches(old_dirs['val'])
old_test  = list_patches(old_dirs['test'])

# --- Gabungan FINAL: Papua lama + gel.4 (train/val/test) + gel.5 + gel.6 (train-only) ---
final_train_files = old_train + gel4_train + train_new_gel5 + train_new_gel6
val_p  = old_val + gel4_val
test_p = old_test + gel4_test
print(f'\nGabungan FINAL -- train: {len(old_train)} (Papua) + {len(gel4_train)} (gel.4) + '
      f'{len(train_new_gel5)} (gel.5) + {len(train_new_gel6)} (gel.6) = {len(final_train_files)}')
print(f'                  val  : {len(old_val)} (Papua) + {len(gel4_val)} (gel.4) = {len(val_p)}')
print(f'                  test : {len(old_test)} (Papua) + {len(gel4_test)} (gel.4) = {len(test_p)}')


def _dist_of(files, desc):
    counts = {c: 0 for c in range(N_CLASSES)}

    def _read_lab(f):
        return np.load(f)['lab']

    with ThreadPoolExecutor(max_workers=32) as exe:
        for lab in tqdm(exe.map(_read_lab, files), total=len(files), desc=desc):
            u, cnt = np.unique(lab, return_counts=True)
            for cls, n in zip(u.tolist(), cnt.tolist()):
                if 0 <= int(cls) < N_CLASSES:
                    counts[int(cls)] += int(n)
    return counts


print('\n=== Distribusi piksel FINAL (Papua + gel.4 + gel.5 + gel.6) per split ===')
dist_summary = {}
for split_name, files in (('train', final_train_files), ('val', val_p), ('test', test_p)):
    dist = _dist_of(files, desc=f'Distribusi {split_name}')
    tot = sum(dist.values()) or 1
    dist_summary[split_name] = {'n_patch': len(files), 'pixel_counts': dist}
    print(f'\n{split_name.upper()} ({len(files)} patch):')
    for c in range(N_CLASSES):
        print(f'  {CLASS_NAMES[c]:<16}: {dist[c]:>14,} px  ({100 * dist[c] / tot:5.2f}%)')

save_json(dist_summary, DIST_DIR / 'distribution_combined_papua_plus_gel4_gel5_gel6.json')

# --- Paket jadi .tar (format SAMA dgn Bahan_Training_Fix) -- v3, siap upload sbg Kaggle Dataset ---
COMBINED_DIR = DRIVE_ROOT / 'Bahan_Training_Fix_Combined_v3'
splits_for_archive = {
    'train': [(f'p{i:06d}.npz', f) for i, f in enumerate(final_train_files)],
    'val':   [(f'p{i:06d}.npz', f) for i, f in enumerate(val_p)],
    'test':  [(f'p{i:06d}.npz', f) for i, f in enumerate(test_p)],
}
archives = create_dataset_archives(splits_for_archive, COMBINED_DIR, n_train_parts=6, max_workers=16)
print(f'\nArsip gabungan FINAL -> {COMBINED_DIR}')
for split_name, paths in archives.items():
    print(f'  {split_name}: {[p.name for p in paths]}')
print('\nLangkah Kaggle: upload folder Bahan_Training_Fix_Combined_v3/ sbg Kaggle Dataset BARU')
print('(superseded Bahan_Training_Fix_Combined v2 yg cuma Papua+gel.4+gel.5 -- pakai v3 ini sekarang),')
print('lalu "Add Input" ke improve_model.ipynb (ENV="kaggle").')
